# Installations

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
from scipy.stats import poisson
from scipy.special import logsumexp  # Import logsumexp


drive.mount('/content/drive')

Mounted at /content/drive


# EM function

In [ ]:
def em_algorithm(data, region_types, n_states=3, max_iter=200, convergence_threshold=1e-8, target_noise_prop=0.15):
    n_cells, n_regions = data.shape

    # Initialize with stronger separation
    target_props = np.array([0.45, 0.45, 0.1])
    mixing_proportions = target_props.copy()

    region_medians = np.median(data, axis=0)
    region_q75 = np.percentile(data, 75, axis=0)
    region_q25 = np.percentile(data, 25, axis=0)

    lambda_params = np.zeros((n_states, n_regions))
    cast_early_mask = region_types == 1
    b6_early_mask = ~cast_early_mask

    # Initialize with stronger separation
    # State 0: CAST > B6 (increased contrast)
    lambda_params[0, cast_early_mask] = region_q75[cast_early_mask] * 1.2  # Increased scaling
    lambda_params[0, b6_early_mask] = region_q25[b6_early_mask] * 0.8  # Decreased scaling

    # State 1: B6 > CAST (increased contrast)
    lambda_params[1, cast_early_mask] = region_q25[cast_early_mask] * 0.8  # Decreased scaling
    lambda_params[1, b6_early_mask] = region_q75[b6_early_mask] * 1.2  # Increased scaling

    # Noise state: global median
    lambda_params[2] = region_medians * 0.5

    # Pre-identify noise cells
    total_signal = np.median(data, axis=1)
    q75_signal = np.percentile(data, 75, axis=1)
    q25_signal = np.percentile(data, 25, axis=1)
    signal_spread = (q75_signal - q25_signal) / (total_signal + 1e-6)

    noise_score = signal_spread * (1 / (total_signal + 1e-6))
    noise_threshold = np.percentile(noise_score, 85)
    noise_candidates = noise_score > noise_threshold

    log_likelihood_old = -np.inf

    for iteration in range(max_iter):
        # E-step
        log_responsibilities = np.zeros((n_cells, n_states))

        for k in range(n_states):
            log_poisson = np.sum(poisson.logpmf(data, lambda_params[k]), axis=1)
            log_responsibilities[:, k] = np.log(mixing_proportions[k]) + log_poisson

        # Stronger noise preference
        log_responsibilities[noise_candidates, 2] += 1.5

        # Balance biological states with increased separation
        for i in range(n_cells):
            if not noise_candidates[i]:
                max_resp = max(log_responsibilities[i, 0], log_responsibilities[i, 1])
                min_resp = min(log_responsibilities[i, 0], log_responsibilities[i, 1])
                if np.isfinite(max_resp) and np.isfinite(min_resp):
                    if max_resp - min_resp > 2.5:  # Increased threshold
                        if log_responsibilities[i, 0] > log_responsibilities[i, 1]:
                            log_responsibilities[i, 1] = max_resp - 2.0
                        else:
                            log_responsibilities[i, 0] = max_resp - 2.0

        # Normalize responsibilities
        log_norm = logsumexp(log_responsibilities, axis=1, keepdims=True)
        log_responsibilities -= log_norm
        responsibilities = np.exp(log_responsibilities)

        if np.any(~np.isfinite(responsibilities)):
            responsibilities = np.nan_to_num(responsibilities, nan=1.0/n_states)

        # M-step with controlled mixing
        raw_proportions = responsibilities.mean(axis=0)
        mixing_proportions = 0.7 * target_props + 0.3 * raw_proportions
        mixing_proportions /= mixing_proportions.sum()

        # Update lambda parameters with stronger separation
        for k in range(n_states):
            weighted_sum = np.sum(responsibilities[:, k][:, np.newaxis] * data, axis=0)
            total_weight = responsibilities[:, k].sum()

            if k < 2:  # Biological states
                new_lambdas = weighted_sum / max(total_weight, 1e-10)

                if k == 0:  # CAST > B6
                    lambda_params[k, cast_early_mask] = 0.8 * new_lambdas[cast_early_mask] + 0.2 * region_q75[cast_early_mask] * 1.2
                    lambda_params[k, b6_early_mask] = 0.8 * new_lambdas[b6_early_mask] + 0.2 * region_q25[b6_early_mask] * 0.8
                else:  # B6 > CAST
                    lambda_params[k, cast_early_mask] = 0.8 * new_lambdas[cast_early_mask] + 0.2 * region_q25[cast_early_mask] * 0.8
                    lambda_params[k, b6_early_mask] = 0.8 * new_lambdas[b6_early_mask] + 0.2 * region_q75[b6_early_mask] * 1.2
            else:  # Noise state
                lambda_params[k] = np.median(new_lambdas) * np.ones_like(new_lambdas)

        # Increased minimum difference for state separation
        min_diff = 0.4  # Increased from 0.3

        # State 0: CAST > B6 with stronger separation
        cast_med_0 = np.median(lambda_params[0, cast_early_mask])
        b6_med_0 = np.median(lambda_params[0, b6_early_mask])
        if (cast_med_0 - b6_med_0) < min_diff:
            adjustment = (min_diff - (cast_med_0 - b6_med_0)) * 0.6
            lambda_params[0, cast_early_mask] += adjustment
            lambda_params[0, b6_early_mask] -= adjustment

        # State 1: B6 > CAST with stronger separation
        cast_med_1 = np.median(lambda_params[1, cast_early_mask])
        b6_med_1 = np.median(lambda_params[1, b6_early_mask])
        if (b6_med_1 - cast_med_1) < min_diff:
            adjustment = (min_diff - (b6_med_1 - cast_med_1)) * 0.6
            lambda_params[1, b6_early_mask] += adjustment
            lambda_params[1, cast_early_mask] -= adjustment

        lambda_params = np.maximum(lambda_params, 1e-6)

        # Check convergence
        log_likelihood_new = np.sum(log_norm)
        if iteration > 0 and log_likelihood_old > -np.inf:
            rel_improvement = abs((log_likelihood_new - log_likelihood_old) / abs(log_likelihood_old))
            if rel_improvement < convergence_threshold:
                print(f"Converged after {iteration + 1} iterations")
                break

        log_likelihood_old = log_likelihood_new

    return mixing_proportions, lambda_params, responsibilities

def analyze_atac_seq_data(data, region_types):
    mixing_proportions, lambda_params, responsibilities = em_algorithm(data, region_types)
    cell_states = np.argmax(responsibilities, axis=1)
    max_probs = responsibilities.max(axis=1)

    cast_early_regions = region_types == 1
    b6_early_regions = ~cast_early_regions

    print("\nRobust State Analysis:")
    print("=====================")

    for i in range(3):
        state_name = "CAST > B6" if i == 0 else "B6 > CAST" if i == 1 else "Noise"
        print(f"\nState {i+1} ({state_name}):")

        # CAST statistics with detailed quantiles
        cast_vals = lambda_params[i, cast_early_regions]
        percentiles = np.percentile(cast_vals, [10, 25, 50, 75, 90])
        print(f"CAST regions:")
        print(f"  Median: {percentiles[2]:.3f}")
        print(f"  IQR: {percentiles[3] - percentiles[1]:.3f}")
        print(f"  10-90 range: [{percentiles[0]:.3f}, {percentiles[4]:.3f}]")
        print(f"  Q1-Q3: [{percentiles[1]:.3f}, {percentiles[3]:.3f}]")

        # B6 statistics with detailed quantiles
        b6_vals = lambda_params[i, b6_early_regions]
        percentiles = np.percentile(b6_vals, [10, 25, 50, 75, 90])
        print(f"B6 regions:")
        print(f"  Median: {percentiles[2]:.3f}")
        print(f"  IQR: {percentiles[3] - percentiles[1]:.3f}")
        print(f"  10-90 range: [{percentiles[0]:.3f}, {percentiles[4]:.3f}]")
        print(f"  Q1-Q3: [{percentiles[1]:.3f}, {percentiles[3]:.3f}]")

        if i < 2:
            diff = (np.median(cast_vals) - np.median(b6_vals)) if i == 0 else (np.median(b6_vals) - np.median(cast_vals))
            print(f"Median difference: {diff:.3f}")

    print(f"\nCell Distribution:")
    for i in range(3):
        state_cells = cell_states == i
        med_prob = np.median(max_probs[state_cells]) if np.any(state_cells) else 0
        iqr_prob = np.percentile(max_probs[state_cells], 75) - np.percentile(max_probs[state_cells], 25) if np.any(state_cells) else 0
        print(f"State {i+1}: {state_cells.mean():.3f} (median confidence: {med_prob:.3f}, IQR: {iqr_prob:.3f})")

    return mixing_proportions, lambda_params, responsibilities, cell_states



# Data processing and visualization

In [ ]:
# Data: (n_cells, n_regions) -> Load as numpy array without cell identifiers
data = pd.read_csv('/content/drive/MyDrive/c_merged_bi_ESC_new.csv', index_col=0).values

# Load the region types, assuming region types are stored in a separate CSV file
# (1 for CAST early, 0 for B6 early)
region_types = np.loadtxt('/content/drive/MyDrive/regions_numeric_merged_bi_ESC_new.csv', delimiter=',', dtype=int)


In [ ]:
# Call the analyze_atac_seq_data with the data
mixing_proportions, lambda_params, responsibilities, cell_states = analyze_atac_seq_data(data=data, region_types=region_types)

<ipython-input-7-416b2e108882>:65: RuntimeWarning: invalid value encountered in subtract
  log_responsibilities -= log_norm



Robust State Analysis:

State 1 (CAST > B6):
CAST regions:
  Median: 0.567
  IQR: 1.198
  10-90 range: [0.273, 3.088]
  Q1-Q3: [0.344, 1.542]
B6 regions:
  Median: 0.086
  IQR: 0.676
  10-90 range: [0.000, 1.573]
  Q1-Q3: [0.000, 0.676]
Median difference: 0.481

State 2 (B6 > CAST):
CAST regions:
  Median: 0.098
  IQR: 0.606
  10-90 range: [0.000, 1.648]
  Q1-Q3: [0.000, 0.606]
B6 regions:
  Median: 0.572
  IQR: 1.322
  10-90 range: [0.248, 3.061]
  Q1-Q3: [0.337, 1.659]
Median difference: 0.474

State 3 (Noise):
CAST regions:
  Median: 0.410
  IQR: 0.000
  10-90 range: [0.410, 0.410]
  Q1-Q3: [0.410, 0.410]
B6 regions:
  Median: 0.410
  IQR: 0.000
  10-90 range: [0.410, 0.410]
  Q1-Q3: [0.410, 0.410]

Cell Distribution:
State 1: 0.341 (median confidence: 0.881, IQR: 0.000)
State 2: 0.584 (median confidence: 0.881, IQR: 0.000)
State 3: 0.075 (median confidence: 1.000, IQR: 0.000)


In [ ]:
cell_states_df = pd.DataFrame(cell_states, columns=['Cell_State'], index=cell_names)

# Save to CSV
cell_states_df.to_csv('/content/drive/MyDrive/cell_states_bi_ESC_new.csv', index=True)

# Control - annotation shuffling

In [ ]:
import numpy as np

# Assuming region_types is a 1D numpy array
first_part = region_types[:len(region_types)//2]  # Get the first half
second_part = region_types[len(region_types)//2:] # Get the second half

# Shuffle the first part
shuffled_first_part = np.random.permutation(first_part)

# Create the shuffled second part based on the shuffled first part
shuffled_second_part = 1 - shuffled_first_part

# Combine the shuffled parts to get the shuffled region_types
shuffled_region_types = np.concatenate([shuffled_first_part, shuffled_second_part])

In [ ]:
mixing_proportions, lambda_params, responsibilities, cell_states = analyze_atac_seq_data(data=data, region_types=shuffled_region_types)

<ipython-input-3-416b2e108882>:65: RuntimeWarning: invalid value encountered in subtract
  log_responsibilities -= log_norm



Robust State Analysis:

State 1 (CAST > B6):
CAST regions:
  Median: 0.554
  IQR: 1.220
  10-90 range: [0.272, 2.974]
  Q1-Q3: [0.354, 1.574]
B6 regions:
  Median: 0.071
  IQR: 0.602
  10-90 range: [0.000, 1.691]
  Q1-Q3: [0.000, 0.602]
Median difference: 0.483

State 2 (B6 > CAST):
CAST regions:
  Median: 0.092
  IQR: 0.654
  10-90 range: [0.000, 1.558]
  Q1-Q3: [0.000, 0.654]
B6 regions:
  Median: 0.565
  IQR: 1.286
  10-90 range: [0.253, 3.206]
  Q1-Q3: [0.334, 1.620]
Median difference: 0.473

State 3 (Noise):
CAST regions:
  Median: 0.420
  IQR: 0.000
  10-90 range: [0.420, 0.420]
  Q1-Q3: [0.420, 0.420]
B6 regions:
  Median: 0.420
  IQR: 0.000
  10-90 range: [0.420, 0.420]
  Q1-Q3: [0.420, 0.420]

Cell Distribution:
State 1: 0.281 (median confidence: 0.881, IQR: 0.000)
State 2: 0.639 (median confidence: 0.881, IQR: 0.000)
State 3: 0.080 (median confidence: 1.000, IQR: 0.000)


In [ ]:
# Assuming 'cell_states' is your vector
cell_states_df = pd.DataFrame(cell_states, columns=['Cell_State'])

# Save to CSV
cell_states_df.to_csv('/content/drive/MyDrive/cell_states_bi_ESC_shuf.csv', index=False)

In [ ]:
# Assuming 'cell_states' is your vector
shuffled_region_types_df = pd.DataFrame(shuffled_region_types, columns=['Cell_State'])

# Save to CSV
shuffled_region_types_df.to_csv('/content/drive/MyDrive/shuffled_region_types.csv', index=False)